# M8 · Backfill `fact_fueling`

**Goal:** load fueling events (fuel-ups) joining `staging.document` (header — date, cost, vehicle) and `staging.fueling` (detail — quantity, odometer, payment).

**SQL file:** `sql/24_fact_fueling_incr.sql`. DELETE-INSERT-on-window (no reliable natural key).

**Watermark column:** `document.date_operation`. We filter `doc_type='Fueling'` at load time.

In [ ]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.config import settings, load_pipeline_config
from accent_fleet.db import get_engine
from sqlalchemy import text
from datetime import datetime, timedelta

s = settings(); cfg = load_pipeline_config()
MIN_TIME = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
print(f'DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}')

## 2. Inputs

In [ ]:
with get_engine().connect() as conn:
    end_time = conn.execute(text("SELECT MAX(date_operation) FROM staging.document WHERE doc_type='Fueling'")).scalar_one()
    n_fuel = conn.execute(text("SELECT COUNT(*) FROM staging.document WHERE doc_type='Fueling'")).scalar_one()
print(f'document rows where doc_type=Fueling: {n_fuel:,}')
print('max date_operation =', end_time)

## 3. Execute

In [ ]:
import importlib, time, pandas as pd
import accent_fleet.transforms.facts as facts_module
from accent_fleet.pipeline.run_log import begin_run, end_run
facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

run_id = begin_run(mode='notebook:10_load_fact_fueling')
progress, cursor, t0 = [], MIN_TIME, time.time()
try:
    while cursor < end_time:
        nxt = min(cursor + timedelta(days=CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_fueling', run_id=run_id, window_end=nxt)
        progress.append({'window_start': cursor, 'window_end': nxt, 'rows_loaded': res.rows_loaded})
        cursor = nxt
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc)); raise
print(f'done in {time.time()-t0:.1f}s')
pd.DataFrame(progress).tail(10)

## 4. Inspect

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_fueling')).scalar_one()
    summary = pd.read_sql(text('''SELECT
        COUNT(*) AS rows_loaded,
        AVG(quantity_l) AS avg_litres,
        AVG(cost_per_litre) AS avg_eur_per_l,
        SUM(cost_total) AS total_cost
      FROM warehouse.fact_fueling'''), conn)
print(f'fact_fueling rows: {total:,}'); display(summary)